# Raw Baseline Kaggle Submission

This is the only supported Kaggle code-submission notebook for the midterm-compliant repo surface.

Fixed Kaggle inputs:
- `/kaggle/input/svg-kaggle-data/`
- `/kaggle/input/qwen25-coder-1p5b-instruct/`
- `/kaggle/input/svg-raw-baseline-adapter/`

This notebook writes only:
- `/kaggle/working/submission.csv`
- `/kaggle/working/submission_debug.csv`

It runs offline, loads the base model plus the raw-baseline adapter, uses deterministic decoding, and enforces the strict `256x256` SVG contract before saving outputs.


In [ ]:
from pathlib import Path
import copy
import io
import random
import re
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
import pandas as pd
import torch
from PIL import Image
from lxml import etree
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

DATA_ROOT = Path("/kaggle/input/svg-kaggle-data")
BASE_MODEL_DIR = Path("/kaggle/input/qwen25-coder-1p5b-instruct")
ADAPTER_DIR = Path("/kaggle/input/svg-raw-baseline-adapter")

TEST_CSV = DATA_ROOT / "test.csv"
SAMPLE_SUBMISSION_CSV = DATA_ROOT / "sample_submission.csv"
WORK_ROOT = Path("/kaggle/working")
SUBMISSION_CSV = WORK_ROOT / "submission.csv"
DEBUG_CSV = WORK_ROOT / "submission_debug.csv"

SVG_MAX_NEW_TOKENS = 1536
SVG_BATCH_SIZE = 128
RENDER_SIZE = 256
MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
STRICT_VIEWBOX = f"0 0 {RENDER_SIZE} {RENDER_SIZE}"
STRICT_BOX = (0.0, 0.0, float(RENDER_SIZE), float(RENDER_SIZE))
SEED = 1337

required_paths = [TEST_CSV, SAMPLE_SUBMISSION_CSV, BASE_MODEL_DIR, ADAPTER_DIR]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(f"Missing Kaggle inputs: {missing_paths}")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

print(f"Seed: {SEED}")
print(f"Base model dir: {BASE_MODEL_DIR}")
print(f"Adapter dir: {ADAPTER_DIR}")
print(f"Test CSV: {TEST_CSV}")
print(f"Sample submission CSV: {SAMPLE_SUBMISSION_CSV}")
print(f"Submission output: {SUBMISSION_CSV}")
print(f"Debug output: {DEBUG_CSV}")


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_DIR, local_files_only=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_DIR,
    device_map="auto",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    local_files_only=True,
)
model = PeftModel.from_pretrained(model, ADAPTER_DIR, local_files_only=True)
model = model.merge_and_unload()
model.eval()

print("Loaded base model plus raw-baseline adapter for deterministic inference.")


In [ ]:
ET.register_namespace("", "http://www.w3.org/2000/svg")

SVG_NS = "http://www.w3.org/2000/svg"
STRICT_SVG_OPEN = f'<svg xmlns="{SVG_NS}" width="{RENDER_SIZE}" height="{RENDER_SIZE}" viewBox="{STRICT_VIEWBOX}">'
FALLBACK_SVG = f'{STRICT_SVG_OPEN}<rect width="{RENDER_SIZE}" height="{RENDER_SIZE}" fill="white"/></svg>'
DIRECT_ROOT_TAGS = {"defs", "title", "desc", "style"}
DEBUG_COLUMNS = [
    "id",
    "reason",
    "gate_reason",
    "source_gate_reason",
    "normalized_gate_reason",
    "normalization_status",
    "strict_contract",
    "strict_issues",
    "collapsed",
    "hit_token_cap",
    "generated_tokens",
    "failure_reasons",
    "extracted_svg",
    "final_svg",
]
ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter",
}
EVENT_HANDLER_RE = re.compile(r"\son[a-zA-Z]+\s*=", re.IGNORECASE)
EXTERNAL_HREF_RE = re.compile(r"\s(?:href|xlink:href)\s*=\s*[\"']\s*(?:https?:|//)", re.IGNORECASE)
REMOTE_REF_RE = re.compile(r"@import\b|url\(\s*[\"']?(?:https?:|//)", re.IGNORECASE)
ROOT_TAG_RE = re.compile(r"<svg\b[^>]*>", flags=re.IGNORECASE | re.DOTALL)


def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def extract_svg(text: str) -> str:
    text = str(text or "").strip()
    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()
    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()
    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()
    return text


def render_svg(svg: str):
    try:
        png = cairosvg.svg2png(bytestring=svg.encode("utf-8"), output_width=RENDER_SIZE, output_height=RENDER_SIZE)
        return np.array(Image.open(io.BytesIO(png)).convert("RGB"))
    except Exception:
        return None


def get_attr_value(opening_tag: str, attr_name: str):
    pattern = rf"(\s{re.escape(attr_name)}\s*=\s*)([\"'])(.*?)\2"
    match = re.search(pattern, opening_tag, flags=re.IGNORECASE | re.DOTALL)
    return None if match is None else match.group(3)


def format_number(value: float) -> str:
    if abs(value - round(value)) < 1e-9:
        return str(int(round(value)))
    text = f"{value:.12g}"
    return text.rstrip("0").rstrip(".") if "e" not in text and "." in text else text


def parse_numeric_attr(value):
    if value is None:
        return None
    match = re.fullmatch(r"\s*([+-]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][+-]?\d+)?)\s*(px)?\s*", str(value))
    if match is None:
        return None
    numeric = float(match.group(1))
    return numeric if numeric > 0 else None


def parse_viewbox(value):
    if value is None:
        return None
    pieces = [piece for piece in re.split(r"[\s,]+", str(value).strip()) if piece]
    if len(pieces) != 4:
        return None
    try:
        x, y, width, height = [float(piece) for piece in pieces]
    except ValueError:
        return None
    return (x, y, width, height) if width > 0 and height > 0 else None


def derive_source_box(root: ET.Element):
    viewbox = parse_viewbox(root.attrib.get("viewBox"))
    if viewbox is not None:
        return viewbox, "viewBox"
    width = parse_numeric_attr(root.attrib.get("width"))
    height = parse_numeric_attr(root.attrib.get("height"))
    if width is not None and height is not None:
        return (0.0, 0.0, width, height), "width_height"
    return STRICT_BOX, "default"


def rebuild_svg_root(root: ET.Element, source_box, source_kind: str):
    new_root = ET.Element(f"{{{SVG_NS}}}svg")
    new_root.set("width", str(RENDER_SIZE))
    new_root.set("height", str(RENDER_SIZE))
    new_root.set("viewBox", STRICT_VIEWBOX)

    source_x, source_y, source_width, source_height = source_box
    needs_transform = source_box != STRICT_BOX
    transform_group = None

    if needs_transform:
        sx = RENDER_SIZE / source_width
        sy = RENDER_SIZE / source_height
        tx = -source_x * sx
        ty = -source_y * sy
        transform_group = ET.Element(f"{{{SVG_NS}}}g")
        transform_group.set(
            "transform",
            f"matrix({format_number(sx)} 0 0 {format_number(sy)} {format_number(tx)} {format_number(ty)})",
        )

    group_inserted = False
    for child in list(root):
        child_copy = copy.deepcopy(child)
        child_tag = strip_namespace(child.tag)
        if child_tag in DIRECT_ROOT_TAGS or transform_group is None:
            new_root.append(child_copy)
            continue
        if not group_inserted:
            new_root.append(transform_group)
            group_inserted = True
        transform_group.append(child_copy)

    status = "unchanged" if not needs_transform else f"scaled_from_{source_kind}"
    return ET.tostring(new_root, encoding="unicode"), status


def canonicalize_to_strict_256(svg: str):
    try:
        root = ET.fromstring(svg)
    except Exception:
        return svg, "parse_failed"
    if strip_namespace(root.tag) != "svg":
        return svg, "root_not_svg"
    source_box, source_kind = derive_source_box(root)
    return rebuild_svg_root(root, source_box, source_kind)


def strict_contract_issues(svg: str) -> list[str]:
    issues = []
    opening_match = ROOT_TAG_RE.search(svg)
    if opening_match is None:
        return ["strict_parse_failed"]
    opening_tag = opening_match.group(0)
    if get_attr_value(opening_tag, "xmlns") != SVG_NS:
        issues.append("missing_xmlns")
    try:
        root = ET.fromstring(svg)
    except Exception:
        return issues + ["strict_parse_failed"]
    if strip_namespace(root.tag) != "svg":
        return issues + ["root_not_svg"]
    if root.attrib.get("width") != str(RENDER_SIZE) or root.attrib.get("height") != str(RENDER_SIZE):
        issues.append("width_height_not_exact")
    if root.attrib.get("viewBox") != STRICT_VIEWBOX:
        issues.append("viewbox_not_exact")
    return issues


def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"
    svg = svg.strip()
    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"
    if EVENT_HANDLER_RE.search(svg):
        return 0, "disallowed_attr:event_handler"
    if EXTERNAL_HREF_RE.search(svg):
        return 0, "disallowed_ref:external_href"
    if REMOTE_REF_RE.search(svg):
        return 0, "disallowed_ref:remote_url"
    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"
    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"
    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)
        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"
        if tag == "path":
            path_count += 1
    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"
    if render_svg(svg) is None:
        return 0, "render_failed"
    return 1, "valid"


def repair_svg(svg: str) -> str:
    if not svg:
        return svg
    svg = svg.strip()
    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]
    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()
    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[: end + len("</svg>")]
    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"
    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)
    svg = re.sub(r"<[^>]*$", "", svg)
    return svg


def recover_svg_with_lxml(svg: str) -> str:
    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg


def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True
    paths = [elem for elem in root.iter() if strip_namespace(elem.tag) == "path"]
    nonempty_paths = [path for path in paths if path.attrib.get("d", "").strip()]
    if paths and not nonempty_paths:
        return True
    total_elems = sum(1 for _ in root.iter())
    return total_elems <= 1


def candidate_from_svg(extracted_svg: str, stage_svg: str, reason: str):
    source_valid, source_gate_reason = validity_gate(stage_svg)
    if source_valid == 0:
        return None, source_gate_reason
    final_svg, normalization_status = canonicalize_to_strict_256(stage_svg)
    normalized_valid, normalized_gate_reason = validity_gate(final_svg)
    if normalized_valid == 0:
        return None, normalized_gate_reason
    strict_issues = strict_contract_issues(final_svg)
    if strict_issues or looks_collapsed(final_svg):
        return None, "strict_contract_failed" if strict_issues else "collapsed"
    return {
        "reason": reason,
        "gate_reason": normalized_gate_reason,
        "source_gate_reason": source_gate_reason,
        "normalized_gate_reason": normalized_gate_reason,
        "normalization_status": normalization_status,
        "strict_contract": True,
        "strict_issues": "",
        "collapsed": False,
        "hit_token_cap": False,
        "generated_tokens": 0,
        "failure_reasons": "",
        "extracted_svg": extracted_svg,
        "final_svg": final_svg,
    }, normalized_gate_reason


def clean_svg_output(raw_text: str):
    extracted_svg = extract_svg(str(raw_text or ""))
    failures = []

    raw_candidate, raw_failure = candidate_from_svg(extracted_svg, extracted_svg, "valid")
    if raw_candidate is not None:
        return raw_candidate
    failures.append(f"raw={raw_failure}")

    repaired_svg = repair_svg(extracted_svg)
    repaired_candidate, repaired_failure = candidate_from_svg(extracted_svg, repaired_svg, "repaired")
    if repaired_candidate is not None:
        repaired_candidate["failure_reasons"] = "|".join(failures)
        return repaired_candidate
    failures.append(f"repaired={repaired_failure}")

    xml_svg = recover_svg_with_lxml(repaired_svg)
    xml_candidate, xml_failure = candidate_from_svg(extracted_svg, xml_svg, "xml")
    if xml_candidate is not None:
        xml_candidate["failure_reasons"] = "|".join(failures)
        return xml_candidate
    failures.append(f"xml={xml_failure}")

    fallback_issues = strict_contract_issues(FALLBACK_SVG)
    return {
        "reason": "fallback",
        "gate_reason": failures[-1].split("=", 1)[-1] if failures else "fallback",
        "source_gate_reason": "",
        "normalized_gate_reason": "",
        "normalization_status": "fallback",
        "strict_contract": not fallback_issues,
        "strict_issues": ",".join(fallback_issues),
        "collapsed": False,
        "hit_token_cap": False,
        "generated_tokens": 0,
        "failure_reasons": "|".join(failures),
        "extracted_svg": extracted_svg,
        "final_svg": FALLBACK_SVG,
    }


fallback_strict_issues = strict_contract_issues(FALLBACK_SVG)
if fallback_strict_issues:
    raise ValueError(f"Fallback SVG is not strict-contract compliant: {fallback_strict_issues}")


In [ ]:
sample_submission_df = pd.read_csv(SAMPLE_SUBMISSION_CSV, keep_default_na=False)
test_df = pd.read_csv(TEST_CSV, keep_default_na=False)
test_df = test_df.dropna(subset=["id", "prompt"]).copy()
test_df["id"] = test_df["id"].astype(str)
test_df["prompt"] = test_df["prompt"].astype(str)
sample_submission_df["id"] = sample_submission_df["id"].astype(str)

if sample_submission_df.columns.tolist() != ["id", "svg"]:
    raise ValueError(f"Expected sample submission columns ['id', 'svg'], found {sample_submission_df.columns.tolist()}")
if sample_submission_df["id"].tolist() != test_df["id"].tolist():
    raise AssertionError("sample_submission.csv ids do not match test.csv ids in order.")


def as_bool(value) -> bool:
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return value != 0
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def build_svg_input(prompt: str) -> str:
    return f"Prompt: {prompt.strip()}\nSVG:\n"


def generate_text_batch(inputs_text, batch_size, max_new_tokens):
    batch_outputs = []
    next_index = 0
    current_batch_size = max(1, batch_size)
    progress_bar = tqdm(total=len(inputs_text))

    while next_index < len(inputs_text):
        batch_inputs_text = inputs_text[next_index:next_index + current_batch_size]
        inputs = None
        outputs = None
        try:
            inputs = tokenizer(
                batch_inputs_text,
                return_tensors="pt",
                padding=True,
                truncation=True,
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )

            input_width = int(inputs["input_ids"].shape[1])
            generated_only = outputs[:, input_width:].detach().cpu()
            decoded_batch = tokenizer.batch_decode(generated_only, skip_special_tokens=True)
            eos_token_id = tokenizer.eos_token_id

            for raw_text, generated_ids in zip(decoded_batch, generated_only):
                generated_token_ids = generated_ids.tolist()
                if eos_token_id is not None and eos_token_id in generated_token_ids:
                    eos_index = generated_token_ids.index(eos_token_id)
                    effective_token_ids = generated_token_ids[:eos_index]
                    hit_token_cap = False
                else:
                    effective_token_ids = generated_token_ids
                    hit_token_cap = len(effective_token_ids) >= max_new_tokens
                batch_outputs.append(
                    {
                        "raw_text": raw_text,
                        "generated_tokens": len(effective_token_ids),
                        "hit_token_cap": hit_token_cap,
                    }
                )

            next_index += len(batch_inputs_text)
            progress_bar.update(len(batch_inputs_text))
        except torch.cuda.OutOfMemoryError:
            if current_batch_size == 1:
                raise
            current_batch_size = max(1, current_batch_size // 2)
            print(f"CUDA OOM. Retrying with batch size {current_batch_size}.")
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        finally:
            del inputs
            del outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    progress_bar.close()
    return batch_outputs


def build_candidate(generation_output: dict):
    candidate = clean_svg_output(generation_output.get("raw_text", ""))
    candidate["generated_tokens"] = int(generation_output.get("generated_tokens", 0))
    candidate["hit_token_cap"] = as_bool(generation_output.get("hit_token_cap", False))
    return candidate


def build_submission(test_df, svg_batch_size=SVG_BATCH_SIZE):
    prompts = test_df["prompt"].astype(str).tolist()
    generation_outputs = generate_text_batch(
        [build_svg_input(prompt) for prompt in prompts],
        batch_size=svg_batch_size,
        max_new_tokens=SVG_MAX_NEW_TOKENS,
    )
    candidates = [build_candidate(output) for output in generation_outputs]

    submission_df = pd.DataFrame(
        {
            "id": test_df["id"].tolist(),
            "svg": [candidate["final_svg"] for candidate in candidates],
        }
    )

    debug_rows = []
    for row, candidate in zip(test_df.itertuples(index=False), candidates):
        debug_row = {column: candidate.get(column, "") for column in DEBUG_COLUMNS if column != "id"}
        debug_row["id"] = row.id
        debug_rows.append(debug_row)
    debug_df = pd.DataFrame(debug_rows)[DEBUG_COLUMNS]
    return submission_df, debug_df


submission_df, debug_df = build_submission(test_df)
submission_df.to_csv(SUBMISSION_CSV, index=False)
debug_df.to_csv(DEBUG_CSV, index=False)

strict_failures = [svg for svg in submission_df["svg"].tolist() if strict_contract_issues(svg)]
if strict_failures:
    raise AssertionError(f"Found {len(strict_failures)} non-strict SVG outputs in submission.csv")

print("Reason counts:")
print(debug_df["reason"].value_counts())
print(f"Strict-contract pass rate: {float(debug_df['strict_contract'].map(as_bool).mean()):.4f}")
print(f"Token-cap hits: {int(debug_df['hit_token_cap'].map(as_bool).sum())}")
print(submission_df.head())
print(debug_df.head())
